In [48]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [49]:
# Create an API Client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [50]:
# Helper Functions
def add_user_message(messages, text):
  user_message = { "role": "user", "content": text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = { "role": "assistant", "content": text}
  messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature
  }

  if system:
    params["system"] = system

  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  message = client.messages.create(**params)

  return message.content[0].text

In [51]:
import json

def generate_dataset():
  prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
      {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
      },
      ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
  """

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```json")
  text = chat(messages, stop_sequences=["```"])
  
  return json.loads(text)

In [52]:
dataset = generate_dataset()

print(dataset)

with open("dataset.json", "w") as f:
  json.dump(dataset, f, indent=2)

[{'task': 'Create a JSON configuration object for an AWS Lambda function that processes S3 events, with environment variables for a DynamoDB table name and log level', 'format': 'json', 'solution_criteria': 'Valid JSON structure with Lambda runtime configuration, S3 event source mapping, environment variables section containing DynamoDB table name and log level, and appropriate IAM role reference'}, {'task': 'Write a Python function that takes an AWS CloudWatch log message string and extracts the timestamp, log level, and message content using regex pattern matching', 'format': 'python', 'solution_criteria': "Function accepts a log string parameter, uses regex to parse CloudWatch log format (e.g., '2024-01-15T10:30:45Z ERROR Connection timeout'), returns a dictionary with 'timestamp', 'level', and 'message' keys with correctly extracted values"}, {'task': 'Create a regex pattern that validates AWS IAM role ARN format (e.g., arn:aws:iam::123456789012:role/MyRole)', 'format': 'regex', 's

In [59]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt = f"""
  Please solve the following task:
  
  {test_case["task"]}

  * Respond only with Python, JSON, or a plain Regex
  * Do not add any comments or commentary or explanation
  """

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```code")
  output = chat(messages, stop_sequences=["```"])
  return output

In [60]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Criteria you should use to evaluate the solution:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>

    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [61]:
# Functions to validate the output structure
import re
import ast

def validate_json(text):
  try:
    json.loads(text.strip())
    return 10
  except json.JSONDecodeError:
    return 0


def validate_python(text):
  try:
    ast.parse(text.strip())
    return 10
  except SyntaxError:
    return 0


def validate_regex(text):
  try:
    re.compile(text.strip())
    return 10
  except re.error:
    return 0


def grade_syntax(response, test_case):
  format = test_case["format"]
  if format == "json":
    return validate_json(response)
  elif format == "python":
    return validate_python(response)
  else:
    return validate_regex(response)

In [62]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the results"""
  output = run_prompt(test_case)

  # Grading
  model_grade = grade_by_model(test_case, output)
  model_score = model_grade["score"]
  reasoning = model_grade["reasoning"]

  syntax_score = grade_syntax(output, test_case)

  score = (model_score + syntax_score) / 2

  return {
    "output": output,
    "test_case": test_case,
    "score": score,
    "reasoning": reasoning
  }

In [63]:
from statistics import mean

def run_eval(dataset):
  """Loads the dataset and calls the run_test_case with each case"""
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  average_score = mean([result["score"] for result in results])
  print(f"Average Score: {average_score}")

  return results

In [64]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

Average Score: 8.333333333333334


In [65]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n  \"FunctionName\": \"s3-event-processor\",\n  \"Runtime\": \"python3.11\",\n  \"Role\": \"arn:aws:iam::ACCOUNT_ID:role/lambda-s3-dynamodb-role\",\n  \"Handler\": \"index.handler\",\n  \"Timeout\": 60,\n  \"MemorySize\": 256,\n  \"Environment\": {\n    \"Variables\": {\n      \"DYNAMODB_TABLE_NAME\": \"s3-events-table\",\n      \"LOG_LEVEL\": \"INFO\"\n    }\n  },\n  \"EventSources\": [\n    {\n      \"Type\": \"S3\",\n      \"Events\": [\"s3:ObjectCreated:*\", \"s3:ObjectRemoved:*\"],\n      \"Filter\": {\n        \"Key\": {\n          \"FilterRules\": [\n            {\n              \"Name\": \"prefix\",\n              \"Value\": \"uploads/\"\n            },\n            {\n              \"Name\": \"suffix\",\n              \"Value\": \".json\"\n            }\n          ]\n        }\n      }\n    }\n  ]\n}\n",
    "test_case": {
      "task": "Create a JSON configuration object for an AWS Lambda function that processes S3 events, with environment variables f